# 03 — Physical feature extraction (the novelty's input)

Partner B, week 2. Before running this on real data, make sure
`pytest tests/test_features_synthetic.py` passes locally (pure NumPy, no GPU needed) —
that's the ground-truth check on CFO/I-Q-imbalance/phase-noise recovery.

In [ ]:
!python -m pytest tests/test_features_synthetic.py -q

In [ ]:
from src.preprocessing.wisig_loader import load_wisig
from src.preprocessing.session_split import load_split, apply_split
from src.features.pipeline import extract_features
from src.utils.config import load_config

config = load_config('configs/default.yaml')
seed = config['seed']
df = load_wisig(config['data']['wisig_root'])
split = load_split(f"data/splits/split_seed{seed}.json")
parts = apply_split(df, split)

feat_cfg = config['features']['cfo']
frames = []
for part_name, part_df in parts.items():
    feats = extract_features(
        part_df, config['data']['sample_rate_hz'],
        repeat_len=feat_cfg['repeat_len'], num_repeats=feat_cfg['num_repeats'],
    )
    feats['split'] = part_name
    frames.append(feats)

import pandas as pd
all_features = pd.concat(frames, ignore_index=True)
all_features.drop(columns=['iq']).to_csv('results/features/features.csv', index=False)
all_features.head()

## Sanity check: do features cluster by device, or by session/receiver?

If they cluster by session/receiver instead of device, the consistency gate would be
learning environment, not hardware — flag this now (see PAPER_NOTES.md limitations).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=all_features, x='cfo_hz', y='iq_amp_imbalance_db', hue='tx_id', legend=False, ax=axes[0])
axes[0].set_title('Colored by device (tx_id)')
sns.scatterplot(data=all_features, x='cfo_hz', y='iq_amp_imbalance_db', hue='session_id', legend=False, ax=axes[1])
axes[1].set_title('Colored by session (day + receiver)')
plt.savefig('results/features/device_vs_session_clustering.png', dpi=150, bbox_inches='tight')
plt.show()